In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score,r2_score

In [4]:
df=pd.read_csv("churn_dataset_advanced.csv")

In [5]:
df.head()

,age,monthly_spend,tenure,plan_type,support_calls,last_login_days,churn
0,21,984,NaN,basic,5.0,20.0,yes
1,36,2498,43.0,premium,NaN,NaN,no
2,34,774,NaN,basic,NaN,10.0,no
3,26,1716,NaN,basic,5.0,NaN,yes
4,44,1048,55.0,premium,5.0,4.0,yes


In [6]:
df.shape

(80, 7)

In [7]:
df.describe()

,age,monthly_spend,tenure,support_calls,last_login_days
count,80.000000,80.000000,39.000000,36.000000,38.000000
mean,37.987500,1397.212500,32.641026,2.611111,16.315789
std,11.351121,626.193336,17.794865,2.032279,8.596370
min,20.000000,323.000000,1.000000,0.000000,1.000000
25%,28.000000,853.250000,17.500000,0.750000,10.250000
50%,37.500000,1416.000000,32.000000,3.000000,17.000000
75%,47.000000,1871.250000,45.500000,4.000000,21.750000
max,59.000000,2498.000000,60.000000,6.000000,30.000000


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              80 non-null     int64  
 1   monthly_spend    80 non-null     int64  
 2   tenure           39 non-null     float64
 3   plan_type        80 non-null     object 
 4   support_calls    36 non-null     float64
 5   last_login_days  38 non-null     float64
 6   churn            80 non-null     object 
dtypes: float64(3), int64(2), object(2)
memory usage: 4.5+ KB


In [9]:
df['tenure'] = df['tenure'].fillna(df['tenure'].mean())
df['last_login_days'] = df['last_login_days'].fillna(df['last_login_days'].mean())

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              80 non-null     int64  
 1   monthly_spend    80 non-null     int64  
 2   tenure           80 non-null     float64
 3   plan_type        80 non-null     object 
 4   support_calls    36 non-null     float64
 5   last_login_days  80 non-null     float64
 6   churn            80 non-null     object 
dtypes: float64(3), int64(2), object(2)
memory usage: 4.5+ KB


In [11]:
df.drop('support_calls', axis=1, inplace=True)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              80 non-null     int64  
 1   monthly_spend    80 non-null     int64  
 2   tenure           80 non-null     float64
 3   plan_type        80 non-null     object 
 4   last_login_days  80 non-null     float64
 5   churn            80 non-null     object 
dtypes: float64(2), int64(2), object(2)
memory usage: 3.9+ KB


In [13]:
df["plan_type"] = df["plan_type"].map({"basic": 0, "premium": 1})
df["churn"] = df["churn"].map({"no": 0, "yes": 1})

In [14]:
df.head()

,age,monthly_spend,tenure,plan_type,last_login_days,churn
0,21,984,32.641026,0,20.000000,1
1,36,2498,43.000000,1,16.315789,0
2,34,774,32.641026,0,10.000000,0
3,26,1716,32.641026,0,16.315789,1
4,44,1048,55.000000,1,4.000000,1


In [15]:
X = df.drop("churn", axis=1)
y = df["churn"]

In [16]:
X_train, X_test, y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
model = LogisticRegression()

In [19]:
model.fit(X_train,y_train)

LogisticRegression()

In [20]:
y_pred= model.predict(X_test)

In [21]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.875


In [22]:
from fastapi import FastAPI
import numpy as np

app = FastAPI()

@app.post("/predict")
def predict(data: dict):
    age = data["age"]
    monthly_spend = data["monthly_spend"]
    tenure = data["tenure"]
    plan_type = 0 if data["plan_type"] == "basic" else 1
    input_data = np.array([[age, monthly_spend, tenure, plan_type]])
    prediction = model.predict(input_data)[0]

    return {"churn": "yes" if prediction == 1 else "no"}

uvicorn main:app --reload

POST /predict

{
  "age": 30,
  "monthly_spend": 900,
  "tenure": 10,
  "plan_type": "basic"
}

{
  "churn": "yes"
}

SyntaxError: invalid syntax (513179696.py, line 17)

In [ ]:
!uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/Users/dhayananthj/Downloads/Test']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [10561] using StatReload
ERROR:    Error loading ASGI app. Could not import module "main".
